# Deploy and benchmark vLLM on Intel Xeon 6

Run this notebook with the standard Jupyter Python (IPython) kernel on a Linux Xeon 6 host with local Docker, Bash, curl, and jq. Deployment, benchmarking, transfer, and comparison run in Bash. Enter a Hugging Face read token in the hidden prompt below after accepting the model’s access terms. The host needs enough memory and disk for the selected model, and the container needs network access to vLLM Recipes and Hugging Face. No host-side vLLM installation is required.

The default is **Llama 3.1 8B Instruct**. Accept its Hugging Face access terms before running the notebook. To switch models, choose a text-generation model with a published Xeon 6 recipe and a context window that can fit 2,048 input tokens plus 128 output tokens. [Xeon 6 recipes](https://recipes.vllm.ai/browse?panel=open&hw=xeon6) · [Model access](https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct)

## 1. Check Docker

Each Bash cell starts a new shell. A local container ID file connects the deployment, readiness, benchmark, and cleanup cells.

In [ ]:
%%bash
set -euo pipefail
command -v curl
command -v jq
docker info --format 'Docker server {{.ServerVersion}}'

## 2. Start the server with the bundled Recipes Tool

The tool runs inside the CPU image. It fetches the published recipe, detects the Xeon 6 resources visible to the container, and starts vLLM with the generated configuration. Docker assigns a free loopback port on the host. Model downloads persist in the mounted cache. [Upstream Docker recipe instructions](https://docs.vllm.ai/en/latest/getting_started/installation/cpu/#serve-with-vllm-recipes)

### Enter a Hugging Face token

The prompt hides your input. The token is passed to Docker through the kernel environment and is omitted from the benchmark export.

In [ ]:
import os
from getpass import getpass

token = getpass("Hugging Face read token: ").strip()
if not token:
    raise ValueError("Enter a Hugging Face token with access to Llama 3.1 8B Instruct.")
os.environ["HF_TOKEN"] = token
del token

In [ ]:
%%bash
set -euo pipefail

IMAGE=vllm/vllm-openai-cpu:v0.34.0-x86_64
MODEL=meta-llama/Llama-3.1-8B-Instruct
HARDWARE=xeon6
HF_CACHE="$HOME/.cache/huggingface"
CID_FILE=.vllm-recipes-benchmark.cid

: "${HF_TOKEN:?Run the Hugging Face token cell first.}"

if [[ -e "$CID_FILE" ]]; then
  echo "A container ID file already exists. Run the cleanup cell before redeploying." >&2
  exit 1
fi

mkdir -p "$HF_CACHE"
docker pull "$IMAGE"
docker run --detach --cidfile "$CID_FILE" \
  --label "vllm.recipes.model=$MODEL" \
  --security-opt seccomp=unconfined \
  --cap-add SYS_NICE \
  --shm-size=8g \
  --mount "type=bind,source=$HF_CACHE,target=/root/.cache/huggingface" \
  --publish 127.0.0.1::8000 \
  --env HF_TOKEN \
  --entrypoint tools/recipes/serve_with_recipe.sh \
  "$IMAGE" --model "$MODEL" --hardware "$HARDWARE"

## 3. Wait for the API

The first startup can take several minutes. This cell checks container status and polls the model endpoint. If startup fails or times out, it prints the tail of the Docker logs.

In [ ]:
%%bash
set -euo pipefail

CONTAINER_ID=$(cat .vllm-recipes-benchmark.cid)
STARTUP_TIMEOUT_S=1800
deadline=$((SECONDS + STARTUP_TIMEOUT_S))

while (( SECONDS < deadline )); do
  status=$(docker inspect --format '{{.State.Status}}' "$CONTAINER_ID")
  if [[ "$status" != running ]]; then
    echo "The container is $status." >&2
    break
  fi
  if binding=$(docker port "$CONTAINER_ID" 8000/tcp 2>/dev/null) &&
    served_model=$(curl --noproxy '*' -fs --max-time 5 "http://$binding/v1/models" |
      jq -er '.data[0].id | select(type == "string" and length > 0)'); then
    echo "Serving $served_model at http://$binding"
    exit 0
  fi
  sleep 5
done

echo "vLLM did not become ready." >&2
docker logs --tail 80 "$CONTAINER_ID"
exit 1

## 4. Run the benchmark

Edit `MAX_CONCURRENCY` in the cell below to set how many requests can be in flight. The default is 1. Use the same value on both instances when comparing them.

The benchmark sends 100 measured requests and 3 warmups, with 2,048 input tokens and 128 output tokens per measured request. **SLA goodput** is the number of successful requests per second that meet both limits in the same request, TTFT ≤ 10,000 ms and TPOT ≤ 100 ms. [vLLM benchmark CLI](https://docs.vllm.ai/en/latest/cli/bench/serve/#goodput)

In [ ]:
%%bash
set -euo pipefail

CONTAINER_ID=$(cat .vllm-recipes-benchmark.cid)
binding=$(docker port "$CONTAINER_ID" 8000/tcp)
served_model=$(curl --noproxy '*' -fsS --max-time 5 "http://$binding/v1/models" |
  jq -er '.data[0].id | select(type == "string" and length > 0)')
tokenizer=$(docker inspect --format '{{index .Config.Labels "vllm.recipes.model"}}' "$CONTAINER_ID")

INPUT_TOKENS=2048
OUTPUT_TOKENS=128
MAX_CONCURRENCY=1  # Edit this value to change the concurrency cap.
TTFT_SLA_MS=10000
TPOT_SLA_MS=100
NUM_PROMPTS=100
NUM_WARMUPS=3
SEED=42
mkdir -p benchmark-results
work_dir=$(mktemp -d benchmark-results/.run.XXXXXX)
trap 'rm -rf -- "$work_dir"' EXIT
remote_file="/tmp/vllm-bench-${work_dir##*.}.json"

docker exec "$CONTAINER_ID" vllm bench serve \
  --backend vllm \
  --base-url http://127.0.0.1:8000 \
  --endpoint /v1/completions \
  --model "$served_model" \
  --tokenizer "$tokenizer" \
  --dataset-name random \
  --random-input-len "$INPUT_TOKENS" \
  --random-output-len "$OUTPUT_TOKENS" \
  --random-range-ratio 0 \
  --ignore-eos \
  --num-prompts "$NUM_PROMPTS" \
  --num-warmups "$NUM_WARMUPS" \
  --request-rate inf \
  --max-concurrency "$MAX_CONCURRENCY" \
  --percentile-metrics ttft,tpot,itl \
  --metric-percentiles 99 \
  --goodput "ttft:$TTFT_SLA_MS" "tpot:$TPOT_SLA_MS" \
  --seed "$SEED" \
  --save-result \
  --result-dir /tmp \
  --result-filename "${remote_file##*/}"

docker cp "$CONTAINER_ID:$remote_file" "$work_dir/metrics.json"
docker exec "$CONTAINER_ID" rm -f "$remote_file"
image_id=$(docker inspect --format '{{.Image}}' "$CONTAINER_ID")
jq -n \
  --arg host "$(hostname)" --arg timestamp "$(date -u +%FT%TZ)" \
  --arg model "$tokenizer" --arg image_id "$image_id" \
  --argjson input "$INPUT_TOKENS" --argjson output "$OUTPUT_TOKENS" \
  --argjson concurrency "$MAX_CONCURRENCY" --argjson prompts "$NUM_PROMPTS" \
  --argjson warmups "$NUM_WARMUPS" --argjson seed "$SEED" \
  --argjson ttft_sla "$TTFT_SLA_MS" --argjson tpot_sla "$TPOT_SLA_MS" \
  --slurpfile metrics "$work_dir/metrics.json" '
  {
    host: $host, timestamp: $timestamp, image_id: $image_id,
    workload: {
      model: $model, dataset: "random", input_tokens: $input,
      output_tokens: $output, max_concurrency: $concurrency,
      requests: $prompts, warmups: $warmups, seed: $seed,
      request_rate: "inf", ignore_eos: true,
      sla_ms: {ttft: $ttft_sla, tpot: $tpot_sla}
    },
    metrics: $metrics[0]
  }' > "$work_dir/result.json"
mv "$work_dir/result.json" benchmark-results/local.json
printf 'Saved %s/benchmark-results/local.json\n' "$PWD"

## 5. Clean up

Run this cell when you finish benchmarking. It removes the container created above. The Hugging Face cache stays on the host.

In [ ]:
%%bash
set -euo pipefail

CID_FILE=.vllm-recipes-benchmark.cid
if [[ -s "$CID_FILE" ]]; then
  docker rm --force "$(cat "$CID_FILE")"
fi
rm -f -- "$CID_FILE"

## 6. Transfer instance B’s result through Jupyter

Run sections 1–5 on both instances. Keep the notebook open on **instance A** for the comparison. Each instance saves its own result at `benchmark-results/local.json`, relative to the notebook’s working directory.

1. **On instance B**, open the Jupyter file browser and navigate to the notebook’s working directory. Open `benchmark-results`, then download `local.json`. In JupyterLab, right-click the file and choose **Download**.
2. **On your computer**, rename the downloaded file to `peer.json`. This file contains instance B’s result.
3. **On instance A**, open the Jupyter file browser and navigate to the `benchmark-results` folder that already contains A’s `local.json`. Open that folder, click **Upload Files**, and select `peer.json` from your computer.
4. **On instance A**, confirm the folder contains both `local.json` (A) and `peer.json` (B). Then run section 7 in A’s notebook.

This transfer uses your browser and existing Jupyter sessions. No SSH credentials are needed.

## 7. Compare results on instance A

Run the cell below on A. The percent change is `(B − A) / A × 100`. Higher SLA goodput and lower latency are better. Rerun the benchmark on both instances to compare exports that include the SLA. The cell checks that the recorded workload and image ID match and that all measured requests completed on both instances. The same image can still fetch a newer published recipe on a later run, and hardware detection can choose different settings on each host.

In [ ]:
%%bash
set -euo pipefail

jq -n -r --slurpfile ref benchmark-results/local.json \
  --slurpfile peer benchmark-results/peer.json '
  def row($label; $a; $b):
    if (($a | type) != "number" or ($b | type) != "number") then
      error("Missing numeric metric: " + $label)
    else
      [$label, $a, $b, (if $a == 0 then "n/a"
        else (((($b - $a) / $a * 10000) | round) / 100 | tostring) + "%"
        end)] | @tsv
    end;
  $ref[0] as $a | $peer[0] as $b |
  if $a.workload != $b.workload then error("Benchmark workloads differ")
  elif $a.workload.sla_ms != {ttft: 10000, tpot: 100}
    then error("Rerun both benchmarks with the configured SLA")
  elif $a.image_id != $b.image_id then error("Docker image IDs differ")
  elif ($a.metrics.completed != $a.workload.requests or
        $b.metrics.completed != $b.workload.requests)
    then error("One or both runs did not complete all measured requests")
  else
    "A: \($a.host) (\($a.timestamp))",
    "B: \($b.host) (\($b.timestamp))",
    (["Metric", "A", "B", "B vs A"] | @tsv),
    row("SLA goodput (req/s)"; $a.metrics.request_goodput; $b.metrics.request_goodput),
    row("P99 TTFT (ms)"; $a.metrics.p99_ttft_ms; $b.metrics.p99_ttft_ms),
    row("P99 TPOT (ms)"; $a.metrics.p99_tpot_ms; $b.metrics.p99_tpot_ms),
    row("P99 ITL (ms)"; $a.metrics.p99_itl_ms; $b.metrics.p99_itl_ms)
  end'